# Exploration & Walkthrough Notebook

This notebook is for **understanding and demoing** the pipeline step by step —
useful in a client call to show exactly what's happening at each stage,
rather than just running `train.py` as a black box.

Sections:
1. Load and inspect the dataset
2. Visualize augmentation (see what the model actually "sees")
3. Build the model and count trainable parameters
4. Run a few training epochs live, inline
5. Run predictions and visualize results

In [ ]:
# Add the src/ folder to Python's import path so we can reuse the same
# code the production scripts use — we never duplicate logic here.
import sys
sys.path.append('../src')

import matplotlib.pyplot as plt
import torch
from torchvision import datasets

from data import build_transforms, get_dataloaders
from model import build_model

## 1. Load and inspect the dataset
Point `DATA_DIR` at your labelled image folder (one sub-folder per class).

In [ ]:
DATA_DIR = '../sample_data'  # change this to your dataset path

raw_dataset = datasets.ImageFolder(DATA_DIR)
print(f'Total images: {len(raw_dataset)}')
print(f'Classes found: {raw_dataset.classes}')

# Count images per class — flags imbalance early, before training.
from collections import Counter
counts = Counter([label for _, label in raw_dataset.samples])
for idx, class_name in enumerate(raw_dataset.classes):
    print(f'  {class_name}: {counts[idx]} images')

## 2. Visualize augmentation
This shows the SAME image after several random augmentation passes —
exactly what the model sees during training. Useful to sanity-check that
augmentation isn't too aggressive (e.g. rotating so much the object
becomes unrecognisable).

In [ ]:
import torchvision.transforms.functional as F

sample_image, _ = raw_dataset[0]
train_transform = build_transforms(image_size=224, train=True)

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    augmented = train_transform(sample_image)
    # Undo normalization just for display purposes
    img_display = augmented.permute(1, 2, 0).numpy()
    img_display = (img_display * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
    ax.imshow(img_display.clip(0, 1))
    ax.set_title(f'Augmented #{i+1}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Build the model and inspect it

In [ ]:
num_classes = len(raw_dataset.classes)
model = build_model('resnet50', num_classes=num_classes, freeze_backbone=True)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,} ({trainable_params/total_params:.1%} of total)')
print('\n(Only the trainable portion updates during training — the rest stays')
print('frozen at its pretrained ImageNet values.)')

## 4. Run a few epochs inline (small demo run)
For a full production run, use `python src/train.py` instead — it has
checkpointing, early stopping, and full logging this quick demo skips.

In [ ]:
train_loader, val_loader, class_to_idx = get_dataloaders(DATA_DIR, batch_size=16)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

for epoch in range(3):  # short demo — increase for real training
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1} done. Last batch loss: {loss.item():.4f}')

## 5. Predict and visualize a few validation images

In [ ]:
idx_to_class = {v: k for k, v in class_to_idx.items()}
model.eval()

images, labels = next(iter(val_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    preds = model(images).argmax(dim=1)

fig, axes = plt.subplots(1, min(5, len(images)), figsize=(15, 3))
for i, ax in enumerate(axes):
    img_display = images[i].cpu().permute(1, 2, 0).numpy()
    img_display = (img_display * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
    ax.imshow(img_display.clip(0, 1))
    true_label = idx_to_class[labels[i].item()]
    pred_label = idx_to_class[preds[i].item()]
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label}', color=color)
    ax.axis('off')
plt.tight_layout()
plt.show()